# **importing the needed libraries**

In [ ]:
!pip install numpy==1.25.2

In [ ]:
!pip install gensim

In [ ]:
pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313509 sha256=833c9038bc97a686c4860787a5156323c84028d07a878c5e9b0075bc296cbb65
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


In [ ]:
import re
import nltk
import string
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.isri import ISRIStemmer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec
from gensim.models import FastText
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import fasttext
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
!pip install camel-tools[full]

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.5/556.5 kB 43.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

# **importing the dataset**

In [ ]:
patient_dr = pd.read_csv("/content/altibbi_specialty_data.csv")

In [ ]:
patient_dr.head()

,specialty_id,name_ar,question_body
0,23,طب عيون,استشاره عيون
1,14,جراحة العظام والمفاصل,السلام عليكم ممكن دكتور مفاصل واعصاب
2,14,جراحة العظام والمفاصل,عندي نقص فيتامين د هل ممكن استخدم معه كالسيوم
3,23,طب عيون,عمليه الحول للكبار
4,14,جراحة العظام والمفاصل,ألم بالكتف الايسر من فترة


In [ ]:
#the data is big so we took 25000 random samples of it
df = patient_dr.drop("name_ar", axis=1)
df = df.sample(n=25000, random_state=42)

In [ ]:
df.head()

,specialty_id,question_body
86128,91,أريد التحدث مع طبيب
60680,25,فيني سمنه
73441,25,اريد نظام رجيم مناسب
45872,23,صداع غثيان
86440,14,الم شديد جداً في فالجنب الايسر من الظهر من ٥ ا...


# **Pre-processing**

***Pre-processing using Iris stemming (NLTK)***

In [ ]:
def preprocessing_patient_stemming(patient_Question):
    # setting a standard
    patient_Question = re.sub(r'[إأآ]', 'ا', patient_Question)
    patient_Question = re.sub(r'ؤ', 'و', patient_Question)
    patient_Question = re.sub(r'ئ', 'ي', patient_Question)
    patient_Question = re.sub(r'ء', '', patient_Question)
    patient_Question = re.sub(r'ة', 'ه', patient_Question)
    # removing harkat
    patient_Question = re.sub(r'[\u064B-\u0652]', '', patient_Question)
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    # tokenization
    tokens = nltk.word_tokenize(patient_Question)
    # Remove stopwords
    ar_stopwords = set(stopwords.words('arabic'))
    without_stop = [word for word in tokens if word not in ar_stopwords]
    # stemming
    stemmer = ISRIStemmer()
    steammed = [stemmer.stem(word) for word in without_stop]
    return steammed

In [ ]:
preprocessed_Question_Stemming = [preprocessing_patient_stemming(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_Stemming))[:100]:
    print(f" Question: {question}")
    print(f" Processed Question: {processed_qu}\n")

 Question: أريد التحدث مع طبيب
 Processed Question: ['ارد', 'حدث', 'طبب']

 Question: فيني سمنه
 Processed Question: ['فين', 'سمن']

 Question: اريد نظام رجيم مناسب
 Processed Question: ['ارد', 'نظم', 'رجم', 'نسب']

 Question: صداع غثيان
 Processed Question: ['صدع', 'غثي']

 Question: الم شديد جداً في فالجنب الايسر من الظهر من ٥ ايام مايفك ولاثانيه مع عدم القدره ع المشي والحركه والنوم رغم المسكنات
 Processed Question: ['الم', 'شدد', 'جدا', 'جنب', 'يسر', 'ظهر', '٥', 'ايم', 'ايف', 'ولاثانيه', 'عدم', 'قدر', 'مشي', 'حرك', 'نوم', 'رغم', 'سكن']

 Question: كيف ازيد طولي ماهو الشي الدي نعمله من شأن يزيد الطول
 Processed Question: ['ازد', 'طول', 'اهو', 'الش', 'الد', 'عمل', 'شان', 'يزد', 'طول']

 Question: اعاني من القرنية المخروطية ولقد عملت عملية تثبيت للقرنية منذ ٣ سنوات ونجحت هل يمكنني الان عمل عملية ليزك لتصحيح بصري وترك النظارات؟
 Processed Question: ['اعا', 'قرن', 'خرط', 'لقد', 'عمل', 'عمل', 'ثبت', 'قرن', '٣', 'سنو', 'نجح', 'يمك', 'الن', 'عمل', 'عمل', 'ليز', 'صحح', 'بصر', 'وتر', 'نظر']



***Pre-processing using porter stemmer***

In [ ]:
!camel_data -i light

The following packages will be installed: 'dialectid-model26', 'morphology-db-msa-s31', 'disambig-mle-calima-msa-r13', 'morphology-db-lev-01', 'disambig-mle-calima-egy-r13', 'morphology-db-egy-r13', 'morphology-db-glf-01', 'morphology-db-msa-r13'
Extracting package 'dialectid-model26': 100% 371M/371M [00:00<00:00, 518MB/s]
Extracting package 'morphology-db-msa-s31': 100% 44.8M/44.8M [00:00<00:00, 445MB/s]
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:00<00:00, 535MB/s]
Extracting package 'morphology-db-lev-01': 100% 10.6M/10.6M [00:00<00:00, 551MB/s]
Extracting package 'disambig-mle-calima-egy-r13': 100% 27.2M/27.2M [00:00<00:00, 555MB/s]
Extracting package 'morphology-db-egy-r13': 100% 67.3M/67.3M [00:00<00:00, 558MB/s]
Extracting package 'morphology-db-glf-01': 100% 7.98M/7.98M [00:00<00:00, 539MB/s]
Extracting package 'morphology-db-msa-r13': 100% 40.5M/40.5M [00:00<00:00, 560MB/s]


In [ ]:
from camel_tools.disambig.mle import MLEDisambiguator
disambig = MLEDisambiguator.pretrained()
def preprocessing_patient_camel(patient_Question):
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    #only returning the lexemma of the word using camel tools morphological analysis(lemmatization)
    analysis_output = disambig.disambiguate(patient_Question.split())
    lexemma = [res.analyses[0].analysis.get('lex', res.word) for res in analysis_output if res.analyses]
    ar_stopwords = set(stopwords.words('arabic'))
    ready= []
    for word in lexemma:
      # setting a standard
      word = re.sub(r'[إأآ]', 'ا', word)
      word = re.sub(r'ؤ', 'و', word)
      word = re.sub(r'ئ', 'ي', word)
      word = re.sub(r'ء', '', word)
      word = re.sub(r'ة', 'ه', word)
      # removing harkat
      word = re.sub(r'[\u064B-\u0652]', '', word)
      if word not in ar_stopwords:
        ready.append(word)
    return ready


In [ ]:
preprocessed_Question_camel = [preprocessing_patient_camel(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_camel))[:100]:
    print(f" Question: {question}")
    print(f" Processed Question: {processed_qu}\n")

 Question: أريد التحدث مع طبيب
 Processed Question: ['اراد', 'تحدث', 'طبيب']

 Question: فيني سمنه
 Processed Question: ['سمنه']

 Question: اريد نظام رجيم مناسب
 Processed Question: ['اراد', 'نظام', 'رجيم', 'مناسب']

 Question: صداع غثيان
 Processed Question: ['صداع', 'غثيان']

 Question: الم شديد جداً في فالجنب الايسر من الظهر من ٥ ايام مايفك ولاثانيه مع عدم القدره ع المشي والحركه والنوم رغم المسكنات
 Processed Question: ['الم', 'شديد', 'جد', 'جنب', 'ايسر', 'ظهر', '٥', 'يوم', 'مايفك', 'ولاثانيه', 'عدم', 'قدره', 'مشي', 'حركه', 'نوم', 'رغم', 'مسكن']

 Question: كيف ازيد طولي ماهو الشي الدي نعمله من شأن يزيد الطول
 Processed Question: ['زاد', 'طول', 'ماهو', 'شي', 'الد', 'عمل', 'شان', 'زاد', 'طول']

 Question: اعاني من القرنية المخروطية ولقد عملت عملية تثبيت للقرنية منذ ٣ سنوات ونجحت هل يمكنني الان عمل عملية ليزك لتصحيح بصري وترك النظارات؟
 Processed Question: ['عانى', 'قرني', 'مخروطي', 'عمل', 'عمليه', 'تثبيت', 'قرني', '٣', 'سنه', 'نجح', 'امكن', 'ان', 'عمل', 'عمليه', 'زكى', 'تصحيح', 'بصر

***Pre-processing using Simple Lemmetation (RE)***

In [ ]:
def preprocessing_patient_RE_lemmatization (patient_Question):
    # setting a standard
    patient_Question = re.sub(r'[إأآ]', 'ا', patient_Question)
    patient_Question = re.sub(r'ؤ', 'و', patient_Question)
    patient_Question = re.sub(r'ئ', 'ي', patient_Question)
    patient_Question = re.sub(r'ء', '', patient_Question)
    patient_Question = re.sub(r'ة', 'ه', patient_Question)
    # removing harkat
    patient_Question = re.sub(r'[\u064B-\u0652]', '', patient_Question)
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    # tokenization
    tokens = nltk.word_tokenize(patient_Question)
    # Remove stopwords
    ar_stopwords = set(stopwords.words('arabic'))
    without_stop = [word for word in tokens if word not in ar_stopwords]
    #lemmatization using regular ewxpressions where we manualy remove the parts of the word that are not from the lexemma
    lemmatized_re = [re.sub(r'(هات|ات|ون|ين|ان|يه|ها|هم|كم|نا|ه|ي|ة|ا)$', '', word) for word in without_stop]
    return lemmatized_re

In [ ]:
preprocessed_Question_RE_lemmatized = [preprocessing_patient_RE_lemmatization(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_RE_lemmatized))[:100]:
  print(f" Question: {question}")
  print(f" Processed Question: {processed_qu}\n")

 Question: أريد التحدث مع طبيب
 Processed Question: ['اريد', 'التحدث', 'طبيب']

 Question: فيني سمنه
 Processed Question: ['فين', 'سمن']

 Question: اريد نظام رجيم مناسب
 Processed Question: ['اريد', 'نظام', 'رجيم', 'مناسب']

 Question: صداع غثيان
 Processed Question: ['صداع', 'غثي']

 Question: الم شديد جداً في فالجنب الايسر من الظهر من ٥ ايام مايفك ولاثانيه مع عدم القدره ع المشي والحركه والنوم رغم المسكنات
 Processed Question: ['الم', 'شديد', 'جد', 'فالجنب', 'الايسر', 'الظهر', '٥', 'ايام', 'مايفك', 'ولاثان', 'عدم', 'القدر', 'المش', 'والحرك', 'والنوم', 'رغم', 'المسكن']

 Question: كيف ازيد طولي ماهو الشي الدي نعمله من شأن يزيد الطول
 Processed Question: ['ازيد', 'طول', 'ماهو', 'الش', 'الد', 'نعمل', 'ش', 'يزيد', 'الطول']

 Question: اعاني من القرنية المخروطية ولقد عملت عملية تثبيت للقرنية منذ ٣ سنوات ونجحت هل يمكنني الان عمل عملية ليزك لتصحيح بصري وترك النظارات؟
 Processed Question: ['اعان', 'القرن', 'المخروط', 'ولقد', 'عملت', 'عمل', 'تثبيت', 'للقرن', '٣', 'سنو', 'ونجحت', 'يمكنن', 'ال'

best preprocessing was using camel tools

In [ ]:
#saving the data to use it neural models notebook
data_dic = {
    'x': [' '.join(tokens) for tokens in preprocessed_Question_camel],
    'y': df['specialty_id'].values}
preprocessed_df = pd.DataFrame(data_dic)


In [ ]:
preprocessed_df.shape

(25000, 2)

In [ ]:
preprocessed_df.to_csv('ready_data.csv', index=False)

#**testing different word representation on logistic regression to know which word representation we should use for the rest**

**Traditional word representation:**

First we have to join the text because the input should be joined i choose camel tools for preprocessing

In [ ]:
join_questions_camel = [' '.join(word) for word in preprocessed_Question_camel]

**BOW Bag of words**

In [ ]:
BOW = CountVectorizer()

***BOW - Bag Of Word with  (camel)***

In [ ]:
BOW_representation = BOW.fit_transform(join_questions_camel)

***splitting***

In [ ]:
X_train_BOW_camel, X_test_BOW_camel, y_train_BOW_camel, y_test_BOW_camel = train_test_split(BOW_representation,df['specialty_id'][:BOW_representation.shape[0]], test_size=0.2 , random_state=42)

**TF_IDF**

In [ ]:
TF_IDF = TfidfVectorizer()

In [ ]:
TF_IDF_representation = TF_IDF.fit_transform(join_questions_camel)

***splitting***

In [ ]:
X_train_TFIDF_camel, X_test_TFIDF_camel, y_train_TFIDF_camel, y_test_TFIDF_camel = train_test_split(TF_IDF_representation, df['specialty_id'][:TF_IDF_representation.shape[0]], test_size=0.2 , random_state=42)

***Logistic Regression - BOW***

In [ ]:
LR_BOW = LogisticRegression()
LR_BOW.fit(X_train_BOW_camel.toarray(), y_train_BOW_camel)
y_pred_LR_BOW = LR_BOW.predict(X_test_BOW_camel.toarray())
print(classification_report(y_test_BOW_camel, y_pred_LR_BOW))

              precision    recall  f1-score   support

          14       0.92      0.87      0.90      1357
          18       0.96      0.90      0.93      1020
          23       0.80      0.83      0.81       810
          25       0.91      0.83      0.87       782
          91       0.75      0.87      0.81      1031

    accuracy                           0.86      5000
   macro avg       0.87      0.86      0.86      5000
weighted avg       0.87      0.86      0.87      5000



***Logistic Regression - TFIDF***

In [ ]:
LR_TFIDF= LogisticRegression()
LR_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_LR_TFIDF = LR_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_LR_TFIDF))

              precision    recall  f1-score   support

          14       0.92      0.90      0.91      1357
          18       0.97      0.89      0.93      1020
          23       0.83      0.83      0.83       810
          25       0.93      0.83      0.88       782
          91       0.74      0.89      0.81      1031

    accuracy                           0.87      5000
   macro avg       0.88      0.87      0.87      5000
weighted avg       0.88      0.87      0.87      5000



**Non-Traditional word representation**

In [ ]:
#unlike traditional word representation, the model doesnt return a vector to each sentance
#thus we need to take the frequancy of each world by searching it in the model, then take its average to form a vector for each sentance
def average_vec (model, sentance_tokens):
  word_representation = []
  for word in sentance_tokens:
    if word in model.wv:
      word_representation.append(model.wv[word])
    else: #if the word is not found appeand a vec of zeros
      word_representation.append(np.zeros(150))
    #calculating the average
    return np.mean(word_representation, axis=0)

***Word2vec with camel***

In [ ]:
!wget https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
!unzip full_uni_cbow_100_twitter.zip

--2025-06-07 06:32:36--  https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ia802803.us.archive.org/15/items/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip [following]
--2025-06-07 06:32:36--  https://ia802803.us.archive.org/15/items/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
Resolving ia802803.us.archive.org (ia802803.us.archive.org)... 207.241.232.113
Connecting to ia802803.us.archive.org (ia802803.us.archive.org)|207.241.232.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 962251835 (918M) [application/zip]
Saving to: ‘full_uni_cbow_100_twitter.zip’

full_uni_cbow_100_t 100%[===================>] 917.67M  12.9MB/s    in 81s     

2025-06-07 06:33:58 (11.3 MB/s) - ‘full_uni_cbow_100_twitter.zip’ saved [9

In [ ]:
from gensim.models import Word2Vec
Word2vec= Word2Vec.load("full_uni_cbow_100_twitter.mdl")

In [ ]:
w2v_embeddings =[]
for x in preprocessed_Question_camel:
  v = average_vec(Word2vec, x)
  if v is not None and v.shape == (Word2vec.vector_size,):
    w2v_embeddings.append(v)

word2vec_representation = np.vstack(w2v_embeddings)

In [ ]:
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(word2vec_representation, df['specialty_id'][:len(word2vec_representation)], test_size=0.2 , random_state=42)

***fasttext with camel***

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

--2025-06-07 06:34:34--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.37.90, 13.35.37.111, 13.35.37.123, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.37.90|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4500982519 (4.2G) [application/octet-stream]
Saving to: ‘cc.ar.300.bin.gz’

cc.ar.300.bin.gz    100%[===================>]   4.19G   272MB/s    in 15s     

2025-06-07 06:34:49 (281 MB/s) - ‘cc.ar.300.bin.gz’ saved [4500982519/4500982519]



In [ ]:
!gunzip cc.ar.300.bin.gz

In [ ]:
#loading the already pretrained fasttext
Fasttext = fasttext.load_model('cc.ar.300.bin')

In [ ]:
#step 1: unlike traditional word representation, the model doesnt return a vector to each sentance
#thus we need to take the frequancy of each world by searching it in the model, then take its average to form a vector for each sentance

def average_vec_ft(model, sentence_tokens):
    word_representation = []
    for word in sentence_tokens:
        try:
            word_representation.append(model.get_word_vector(word))
        except: #if the word is not found appeand a vec of zeros
            word_representation.append(np.zeros(model.get_dimension()))
    #calculating the average
    return np.mean(word_representation, axis=0) if word_representation else np.zeros(model.get_dimension())

In [ ]:
#averaging the vectors of fasttext
ft_embeddings =[]
for x in preprocessed_Question_camel:
  v = average_vec_ft(Fasttext, x)
  if v is not None and v.shape == (Fasttext.get_dimension(),):
    ft_embeddings.append(v)

In [ ]:
#making sure that only the labels of the valid fasttext embeddings are saved
ft_labels = []
for x,w in enumerate(preprocessed_Question_camel):
  v = average_vec_ft(Fasttext, w)
  if v is not None and v.shape == (Fasttext.get_dimension(),):
    ft_labels.append(df['specialty_id'].iloc[x])

In [ ]:
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split( np.vstack(ft_embeddings), ft_labels, test_size=0.2, random_state=42)

**Logistic Regression - word2vec**

In [ ]:
LR_w2v = LogisticRegression(max_iter = 1000)
LR_w2v.fit(X_train_w2v, y_train_w2v)
y_pred_LR_w2v = LR_w2v.predict(X_test_w2v)
print(classification_report(y_test_w2v, y_pred_LR_w2v))

              precision    recall  f1-score   support

          14       0.28      0.82      0.41      1138
          18       0.26      0.07      0.11       876
          23       0.11      0.02      0.04       663
          25       0.17      0.01      0.02       637
          91       0.21      0.09      0.12       814

    accuracy                           0.26      4128
   macro avg       0.21      0.20      0.14      4128
weighted avg       0.22      0.26      0.17      4128



**Logistic Regression- FastText**

In [ ]:
LR_FT = LogisticRegression(max_iter = 1000)
LR_FT.fit(X_train_ft, y_train_ft)
y_pred_LR_FT = LR_FT.predict(X_test_ft)
print(classification_report(y_test_ft, y_pred_LR_FT, zero_division=0))

              precision    recall  f1-score   support

          14       0.86      0.87      0.86      1357
          18       0.93      0.86      0.89      1020
          23       0.76      0.74      0.75       810
          25       0.87      0.81      0.84       782
          91       0.73      0.82      0.78      1031

    accuracy                           0.83      5000
   macro avg       0.83      0.82      0.82      5000
weighted avg       0.83      0.83      0.83      5000



# *Best results were using tf-idf*

# **Traditional machine learning models classification**

***Naive Bayes - camel - TFIDF***

In [ ]:
NB_TFIDF = GaussianNB()
NB_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_NB_TFIDF = NB_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_NB_TFIDF))

              precision    recall  f1-score   support

          14       0.76      0.37      0.50      1357
          18       0.65      0.79      0.71      1020
          23       0.59      0.52      0.55       810
          25       0.36      0.84      0.51       782
          91       0.75      0.43      0.54      1031

    accuracy                           0.56      5000
   macro avg       0.62      0.59      0.56      5000
weighted avg       0.65      0.56      0.56      5000



***Decision Tree - camel - TFIDF***

In [ ]:
DT_TFIDF = DecisionTreeClassifier()
DT_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_DT_TFIDF = DT_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_DT_TFIDF))

              precision    recall  f1-score   support

          14       0.84      0.84      0.84      1357
          18       0.91      0.87      0.89      1020
          23       0.70      0.76      0.73       810
          25       0.83      0.79      0.81       782
          91       0.73      0.75      0.74      1031

    accuracy                           0.81      5000
   macro avg       0.80      0.80      0.80      5000
weighted avg       0.81      0.81      0.81      5000



***Logistic Regression - TFIDF***

In [ ]:
LR_TFIDF= LogisticRegression()
LR_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_LR_TFIDF = LR_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_LR_TFIDF))

              precision    recall  f1-score   support

          14       0.92      0.90      0.91      1357
          18       0.97      0.89      0.93      1020
          23       0.83      0.83      0.83       810
          25       0.93      0.83      0.88       782
          91       0.74      0.89      0.81      1031

    accuracy                           0.87      5000
   macro avg       0.88      0.87      0.87      5000
weighted avg       0.88      0.87      0.87      5000

